# 07 — Day 7 Feature Extraction: WavLM-base-plus (all layers)

**Standalone**, same reasoning as Day 4/5: self-contained, only needs the raw dataset, so it can run on
either Kaggle account without depending on another notebook's saved state.

**Why WavLM:** Days 2-3 already compared two SSL models trained on ASR-adjacent objectives
(wav2vec2's contrastive masking, HuBERT's cluster-prediction) and found HuBERT ahead (0.611 vs.
0.581 best-layer). WavLM is a third, distinct pretraining objective: **masked speech denoising** —
trained to reconstruct clean speech from a mix of overlapping/noisy input, which the literature
documents as giving it explicitly stronger noise robustness and greater sensitivity to
paralinguistic and speaker-identity cues than HuBERT (WavLM tops the SUPERB benchmark, including
beating HuBERT-base on diarization-adjacent tasks — see `PROJECT_PLAN.md`'s Day 7+ roadmap for the
full rationale). Same kind of embedding-family bet that made Day 3 (HuBERT over wav2vec2) the
project's first confirmed win.

**Checkpoint:** `microsoft/wavlm-base-plus` — 12 transformer layers, 768 hidden dim, same
architecture shape as wav2vec2-base/HuBERT-base, chosen specifically to keep the layer-for-layer
comparison size-fair (`wavlm-large` has more capacity but a much larger overfitting/compute cost at
only 14 speakers, so it's deliberately not used here).

**What this notebook does:** identical extraction shape to `01_features.ipynb` section 8 (HuBERT
all-layers) — one frozen forward pass per recording with `output_hidden_states=True`, mean+std pool
every one of the 13 hidden states (embedding output + 12 transformer layers), no re-computation
needed to get all layers since they're already produced internally by one forward pass. Saved to
`wavlm_base_plus_all_layers.parquet`. Layer-by-layer evaluation happens in the companion notebook,
`07_evaluate_wavlm.ipynb`.

**Inputs needed:** just the `Fatigue_data` dataset (`project_data/`) — no other notebook's output.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Same robust resolution strategy as every extraction notebook -- search recursively rather than
# assume a fixed /kaggle/input/ depth (this broke once already for datasets, see 01_features.ipynb).
DATA_DIR = next(Path("/kaggle/input").rglob("project_data"))
AUDIO_DIR = DATA_DIR / "audio"
LABELS_CSV = DATA_DIR / "labels.csv"

OUT_DIR = Path("/kaggle/working/features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000

print(f"DATA_DIR   = {DATA_DIR}")
print(f"AUDIO_DIR  = {AUDIO_DIR}")
print(f"LABELS_CSV = {LABELS_CSV}")

In [ ]:
labels = pd.read_csv(LABELS_CSV)
missing = [f for f in labels["filename"] if not (AUDIO_DIR / f).exists()]
assert not missing, f"{len(missing)} filenames missing audio files: {missing[:5]}"
assert len(labels) == 280, f"expected 280 rows, got {len(labels)}"
print(f"Loaded {len(labels)} labeled recordings, all resolve to audio files.")

## 1. Load WavLM (all layers)

`Wav2Vec2FeatureExtractor` is reused for WavLM checkpoints too — WavLM doesn't have its own
dedicated feature-extractor class in `transformers`; preprocessing (raw waveform, zero-mean
unit-variance normalization) is identical to wav2vec2/HuBERT, same as HuBERT's extraction in
`01_features.ipynb` section 8 already established.

In [ ]:
!pip install -q transformers

In [ ]:
import librosa
import torch
from transformers import Wav2Vec2FeatureExtractor, WavLMModel

WAVLM_MODEL_NAME = "microsoft/wavlm-base-plus"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_MODEL_NAME)
model = WavLMModel.from_pretrained(WAVLM_MODEL_NAME).to(device)
model.eval()  # frozen, same as every other extraction notebook in this project

## 2. All-Layer Extraction

Identical shape to `01_features.ipynb`'s `extract_all_layer_embeddings_hubert` — one forward pass
with `output_hidden_states=True`, mean+std pool every hidden state. `microsoft/wavlm-base-plus` has
12 transformer layers (`N_LAYERS = 13`: embedding output + 12 transformer layers), matching
wav2vec2-base and HuBERT-base exactly, so layer indices are directly comparable across all three
models' results.

In [ ]:
N_LAYERS = 13  # embedding output (layer 0) + 12 transformer layers (layers 1-12)


def extract_all_layer_embeddings_wavlm(wav_path: Path) -> np.ndarray:
    """Frozen WavLM forward pass -> every hidden layer -> mean+std pool.
    Returns shape (13, 1536), same layout as the HuBERT/wav2vec2 all-layer files."""
    y, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
    y_trimmed, _ = librosa.effects.trim(y, top_db=25)

    inputs = feature_extractor(y_trimmed, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        outputs = model(input_values, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # tuple of 13 tensors, each (1, T, 768)

    layer_embeddings = []
    for h in hidden_states:
        h = h.squeeze(0)  # (T, 768)
        mean_pool = h.mean(dim=0)
        std_pool = h.std(dim=0)
        layer_embeddings.append(torch.cat([mean_pool, std_pool]).cpu().numpy())  # (1536,)
    return np.stack(layer_embeddings)  # (13, 1536)

In [ ]:
from tqdm.auto import tqdm

wavlm_layer_rows = []
for filename in tqdm(labels["filename"], desc="Extracting WavLM all-layer embeddings"):
    layer_emb = extract_all_layer_embeddings_wavlm(AUDIO_DIR / filename)  # (13, 1536)
    row = {"filename": filename}
    for L in range(N_LAYERS):
        row.update({f"layer{L}_emb_{i}": v for i, v in enumerate(layer_emb[L])})
    wavlm_layer_rows.append(row)

wavlm_layer_embeddings_df = pd.DataFrame(wavlm_layer_rows)
print(wavlm_layer_embeddings_df.shape)  # expect (280, 1 + 13*1536)

## 3. Cache & Verify

In [ ]:
wavlm_out_path = OUT_DIR / "wavlm_base_plus_all_layers.parquet"
wavlm_layer_embeddings_df.to_parquet(wavlm_out_path, index=False)

assert len(wavlm_layer_embeddings_df) == 280, f"expected 280 rows, got {len(wavlm_layer_embeddings_df)}"
n_layer_cols = sum(c.startswith("layer") for c in wavlm_layer_embeddings_df.columns)
assert n_layer_cols == N_LAYERS * 1536, f"expected {N_LAYERS * 1536} layer columns, got {n_layer_cols}"
assert set(wavlm_layer_embeddings_df["filename"]) == set(labels["filename"]), "filename mismatch vs labels.csv"

print(f"Saved {wavlm_out_path}")
print(f"Shape: {wavlm_layer_embeddings_df.shape} ({N_LAYERS} layers x 1536 dims + filename)")
print("All verification checks passed.")

## Next steps

1. **Save this notebook's version.**
2. Open `07_evaluate_wavlm.ipynb`, attach **both** this notebook's output (`wavlm_base_plus_all_layers.parquet`)
   **and** the `Fatigue_data` dataset. It evaluates each of the 13 layers independently under the
   identical nested LOSO/GroupKFold(6) protocol used everywhere else in the project, and compares
   the resulting curve against wav2vec2's (Day 2) and HuBERT's (Day 3) already-logged per-layer results.